# 01 — Manifold Reconstruction and Eigenspectrum Gate

**v1.1 PU Manifold Curvature.** This notebook is a committed deliverable (shipped WITH
outputs, from a deliberate Restart-and-Run-All — see §0.4). Phase 1 owns §0-§5 of this
file; Phase 2 appends §6 onward, described further in §0.5.

Phase 1 (this plan, 01-01) writes §0-§1: environment/reproducibility, dependency install,
and a permanent smoke-config end-to-end self-test proving every layer of the pipeline
(HF config load → seeded subsample → row-alignment assert → L2 normalize → npz cache →
npz read-back → connectivity check → Isomap fit → joblib cache → joblib read-back) works
before the ~1 GB analysis artifact is ever built.

## §0. Environment & Reproducibility

### §0.1 Python floor

In [1]:
import sys
from pathlib import Path

# Make the notebook-local pu_manifold package importable regardless of how the kernel
# was started (D-01 key link: plain relative import, never installed, never imported
# from src/effdim/).
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

assert sys.version_info >= (3, 11), (
    "This notebook requires Python >= 3.11. pyproject.toml declares >=3.8, which is "
    "stale core-dependency drift tracked as LIB-03: scikit-learn 1.9.0 and the "
    "torch/datasets wheels this notebook installs are the real constraint, not the "
    "declared floor. If you are running on Python 3.10 or earlier, create a separate "
    "notebook kernel for this milestone rather than downgrading the pins in "
    "requirements-notebooks.txt."
)
print(f"Python {sys.version.split()[0]} OK (>= 3.11 required)")

Python 3.14.6 OK (>= 3.11 required)


### §0.2 Dependencies

No `## Package Legitimacy Audit` table exists anywhere under `.planning/` (verified
during planning: `grep -rn "Package Legitimacy" .planning/` returns nothing). Per the
documented fallback policy, every package this phase installs is therefore treated as
`[ASSUMED]` and required human confirmation on the registry before install — this was
Task 1 of the Phase 1 plan (01-01), a `checkpoint:human-verify` with
`gate="blocking-human"` that is **never** auto-approvable. Task 1 was reviewed and
**approved**: `torch==2.13.0+cpu` (PyTorch, `+cpu` wheel from
`https://download.pytorch.org/whl/cpu`), `datasets==5.0.1` (Hugging Face `datasets`),
and `matplotlib==3.11.1` were each confirmed as the legitimate, expected project on
PyPI before this cell was authored or run. `numpy`/`scipy`/`scikit-learn`/`faiss-cpu`
are already core `effdim` dependencies and are deliberately not re-pinned here;
`huggingface_hub`/`hf_xet`/`pyarrow` arrive transitively via `datasets`.

In [2]:
%pip install -q -r requirements-notebooks.txt

Note: you may need to restart the kernel to use updated packages.


### §0.3 Reproducibility header

In [3]:
import subprocess

import numpy as np
import scipy
import sklearn
import faiss
import datasets
import torch

# Defined once here (needed for this header) and restated verbatim in §1.1 so that
# section is self-contained for a reader who jumps straight to §1.
SEED = 20260729

git_sha = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()

print("=== Reproducibility header ===")
print(f"SEED           = {SEED}")
print(f"numpy          = {np.__version__}")
print(f"scipy          = {scipy.__version__}")
print(f"scikit-learn   = {sklearn.__version__}")
print(f"faiss          = {faiss.__version__}")
print(f"datasets       = {datasets.__version__}")
print(f"torch          = {torch.__version__}")
print(f"git commit SHA = {git_sha}")

=== Reproducibility header ===
SEED           = 20260729
numpy          = 2.5.1
scipy          = 1.18.0
scikit-learn   = 1.9.0
faiss          = 1.14.3
datasets       = 5.0.1
torch          = 2.13.0+cpu
git commit SHA = ebea110


The git commit SHA is printed for provenance but is **deliberately NOT** part of any
cache key (D-14) — a docstring-only commit must not invalidate a ~1 GB Isomap artifact.
`sklearn`/`numpy`/`scipy` versions, by contrast, **are** part of the fit cache key,
because a library upgrade can silently change numerical results.

### §0.4 Output hygiene policy

1. This notebook is committed **with outputs intact**, produced by a deliberate
   Restart-and-Run-All. It is not run through `nbconvert --clear-output`.
2. `nbstripout` is not installed and no `.gitattributes` filter is added. This repo has
   no pre-commit framework, and adding one for a milestone this size is unwarranted
   ceremony; `nbstripout` is the documented later upgrade path *if* output-diff noise
   ever becomes a real problem, not something adopted pre-emptively.
3. No cell output may embed a bulk array. Outputs here are limited to plots, tables, and
   scalar prints — never a repr of the 10000x768 embedding arrays and never the fitted
   `Isomap` object — so the notebook JSON stays small enough to diff.
4. `.ipynb_checkpoints/` and `notebooks/.cache/` are already present in `.gitignore`
   (verified), so no gitignore change is needed for this phase.

Phase 2, which appends to this same file, inherits this policy rather than re-deciding
it.

### §0.5 Section numbering

Phase 1 owns `§0`-`§5` of this notebook; Phase 2 appends `§6` onward. Existing
sections are never renumbered once written. Subsections use `### §N.M Title`. This
convention, together with D-01's three fixed notebook filenames, is **costly** to change
later: every cache path, cross-notebook check, and doc reference in this milestone is
written against it.

## §1. Smoke-Config End-to-End Self-Test

This section is a **permanent** self-test, not scaffolding to be deleted later: it
exercises every layer this phase touches — HF config load, seeded subsample, row-
alignment assert, L2 normalization, npz cache, npz read-back identity, connectivity
check, deterministic Isomap fit, joblib cache, joblib read-back identity — at a cheap
smoke size (`n_rows=500`) in seconds, before the ~10 minute, ~1 GB analysis fit is ever
paid for.

### §1.1 Configs

In [4]:
# Restated from §0.3 (same value) so this section is self-contained.
SEED = 20260729

SMOKE_CFG = {
    "dataset": "legacysurvey_dinov3_vitb16",
    "seed": SEED,
    "n_rows": 500,
    "normalize": True,
    "n_neighbors": 10,
    "n_components": 4,
    "eigen_solver": "dense",
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "scipy_version": scipy.__version__,
}

ANALYSIS_CFG = {
    "dataset": "legacysurvey_dinov3_vitb16",
    "seed": SEED,
    "n_rows": 10_000,
    "normalize": True,
    "n_neighbors": None,  # derived by a later plan's connectivity sweep (ISO-01/ISO-02)
    "n_components": None,  # derived in a later plan via the D-12 ceil(median(...)) rule
    "eigen_solver": "dense",
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "scipy_version": scipy.__version__,
}

print("SMOKE_CFG:   ", SMOKE_CFG)
print("ANALYSIS_CFG:", ANALYSIS_CFG)

SMOKE_CFG:    {'dataset': 'legacysurvey_dinov3_vitb16', 'seed': 20260729, 'n_rows': 500, 'normalize': True, 'n_neighbors': 10, 'n_components': 4, 'eigen_solver': 'dense', 'sklearn_version': '1.9.0', 'numpy_version': '2.5.1', 'scipy_version': '1.18.0'}
ANALYSIS_CFG: {'dataset': 'legacysurvey_dinov3_vitb16', 'seed': 20260729, 'n_rows': 10000, 'normalize': True, 'n_neighbors': None, 'n_components': None, 'eigen_solver': 'dense', 'sklearn_version': '1.9.0', 'numpy_version': '2.5.1', 'scipy_version': '1.18.0'}


`SMOKE_CFG["n_components"] = 4` is a smoke-test constant only — it is **not** the
analysis `n_components`, which a later plan derives from the D-12 rule
(`ceil(median(...))` over the 8 geometric/intrinsic `effdim.compute_dim` keys). Because
`ANALYSIS_CFG` differs from `SMOKE_CFG` in `n_rows` (and, once derived, `n_components`),
the two configs produce different `config_key` hashes by construction — itself the
ISO-05 "a config change produces a new cache key" property, demonstrated concretely in
§1.3 below rather than merely asserted in prose.

### §1.2 Subsample and alignment (smoke)

In [5]:
from pu_manifold import load_subsample, assert_alignment, config_key, joblib_cache

smoke_data = load_subsample(SMOKE_CFG)
alignment_stats = assert_alignment(
    smoke_data["hsc"],
    smoke_data["legacysurvey"],
    smoke_data["row_indices"],
    seed=SEED,
)

print("hsc shape:         ", smoke_data["hsc"].shape)
print("legacysurvey shape:", smoke_data["legacysurvey"].shape)
print("row_indices shape: ", smoke_data["row_indices"].shape)
print("alignment stats:   ", alignment_stats)

README.md:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

legacysurvey/legacysurvey_dinov3_vitb16.(…): reconstructing file:   0%|          |  0.00B /  580MB            

legacysurvey/legacysurvey_dinov3_vitb16.(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

hsc shape:          (500, 768)
legacysurvey shape: (500, 768)
row_indices shape:  (500,)
alignment stats:    {'s_true': 0.8455923572483339, 'mu_perm': 0.7171236552581232, 'sd_perm': 0.0030748871782488546, 'z': 41.779972578822736, 'margin_z': 5.0, 'n_permutations': 50, 'row_indices_sha256': '3c22a0439ed2838d364e1d1715afd1f815b5cb7367e40450a95822e09239e035'}


### §1.3 Cache round-trip (smoke)

In [6]:
smoke_data_2 = load_subsample(SMOKE_CFG)
for key in smoke_data:
    assert np.array_equal(smoke_data[key], smoke_data_2[key]), (
        f"{key} differs between the two load_subsample(SMOKE_CFG) calls -- cache "
        f"round-trip is not bit-identical."
    )
print("CACHE HIT: second load_subsample(SMOKE_CFG) call returned bit-identical arrays")

smoke_cfg_changed = dict(SMOKE_CFG, n_rows=SMOKE_CFG["n_rows"] + 1)
key_before = config_key(SMOKE_CFG)
key_after = config_key(smoke_cfg_changed)
print(f"config_key(SMOKE_CFG)               = {key_before}")
print(f"config_key(SMOKE_CFG with n_rows+1) = {key_after}")
assert key_before != key_after, "changing n_rows must change the cache key (ISO-05)"
print("ISO-05: a single config field change demonstrably produces a new cache key")

CACHE HIT: second load_subsample(SMOKE_CFG) call returned bit-identical arrays
config_key(SMOKE_CFG)               = a42e1c3087ee5736
config_key(SMOKE_CFG with n_rows+1) = bf4f9657c2b208ed
ISO-05: a single config field change demonstrably produces a new cache key


### §1.4 Connectivity and Isomap fit (smoke)

In [7]:
from scipy.sparse.csgraph import connected_components
from sklearn.manifold import Isomap
from sklearn.neighbors import kneighbors_graph

X_smoke = smoke_data["legacysurvey"]

smoke_graph = kneighbors_graph(
    X_smoke, n_neighbors=SMOKE_CFG["n_neighbors"], mode="distance"
)
n_components_graph, _ = connected_components(smoke_graph, directed=False)
print(f"connected components (smoke, k={SMOKE_CFG['n_neighbors']}): {n_components_graph}")
assert n_components_graph == 1, (
    f"Smoke k-NN graph has {n_components_graph} connected components, not 1. "
    f"sklearn.manifold.Isomap does not raise on a disconnected graph -- it silently "
    f"bridges components with fabricated long edges (PITFALLS Pitfall 1) -- so this "
    f"check must halt rather than let the fit proceed."
)


def _fit_smoke_isomap():
    model = Isomap(
        n_neighbors=SMOKE_CFG["n_neighbors"],
        n_components=SMOKE_CFG["n_components"],
        eigen_solver="dense",
        n_jobs=-1,
    )
    model.fit(X_smoke)
    return model


# eigen_solver="dense" is pinned explicitly here and everywhere in this milestone:
# Isomap has no random_state, and "auto"/"arpack" uses ARPACK with a random start
# vector, so "dense" is what makes the fit deterministic LAPACK (D-15).
smoke_fit_key = config_key(SMOKE_CFG)
isomap_smoke = joblib_cache(f"isomap_{smoke_fit_key}", SMOKE_CFG, _fit_smoke_isomap)

print("dist_matrix_ shape:", isomap_smoke.dist_matrix_.shape)
print("embedding_ shape:  ", isomap_smoke.embedding_.shape)

connected components (smoke, k=10): 1


dist_matrix_ shape: (500, 500)
embedding_ shape:   (500, 4)


The full classical-MDS eigenspectrum audit and the PASS/MARGINAL/FAIL gate
(SPEC-01 through SPEC-07) are **Phase 2's** deliverable and are deliberately absent from
this notebook's §0-§5.

### §1.5 Joblib read-back (smoke)

In [8]:
isomap_smoke_reloaded = joblib_cache(f"isomap_{smoke_fit_key}", SMOKE_CFG, _fit_smoke_isomap)

assert isomap_smoke_reloaded.dist_matrix_.shape == (
    SMOKE_CFG["n_rows"],
    SMOKE_CFG["n_rows"],
), "dist_matrix_ shape does not match the smoke n_rows x n_rows expectation"
assert np.array_equal(
    isomap_smoke_reloaded.embedding_, isomap_smoke.embedding_
), "embedding_ differs between the fresh fit and the cached joblib reload"

print(f"dist_matrix_ shape (reloaded): {isomap_smoke_reloaded.dist_matrix_.shape}")
print("CACHE HIT: joblib_cache reload returned a bit-identical embedding_")

dist_matrix_ shape (reloaded): (500, 500)
CACHE HIT: joblib_cache reload returned a bit-identical embedding_


`notebooks/.cache/` contents are **trusted-local-only**. `joblib.load` is pickle
deserialization (threat T-01-01) — a `.joblib` file obtained from any third party must
never be placed in that directory. `joblib_cache` only ever loads a path it composed
itself from `CACHE_DIR` plus a `config_key` it just computed; there is no helper anywhere
in `pu_manifold` that loads a caller-supplied absolute path.